In [2]:
import pandas as pd
import numpy as np

df_vendas=pd.read_csv("vendas_tech.csv", encoding="utf-8", sep=',', low_memory=False) ##sep=SEPARADOR
df_gerentes=pd.read_excel("gerentes_lojas.xlsx")


In [3]:
##INSPEÇÃO
#display(df_vendas.head(15)) ##PRIMEIRAS
#display(df_vendas.tail(15)) ##ULTIMAS
#display(df_vendas.sample(15)) ##AMOSTRA/LINHAS ALEATÓRIAS
#display(df_vendas.shape)  ##(LINHAS, COLUNAS)
#display(df_vendas.columns) ##[COLUNAS]
df_vendas.info() ##INFORMAÇÕES
display(df_vendas.describe()) ##DESCRIÇÃO


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100100 entries, 0 to 100099
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   ID_Pedido       100100 non-null  int64  
 1   Data            100100 non-null  object 
 2   Loja            98099 non-null   object 
 3   Produto         100100 non-null  object 
 4   Preco_Unitario  100100 non-null  float64
 5   Qtd             100100 non-null  int64  
 6   Cliente         100100 non-null  object 
 7   Data_Base       1 non-null       object 
dtypes: float64(1), int64(2), object(5)
memory usage: 6.1+ MB


,ID_Pedido,Preco_Unitario,Qtd
count,100100.000000,100100.000000,100100.000000
mean,50004.810180,2000.204595,1.499101
std,28866.872543,1841.050733,1.241605
min,1.000000,40.000000,1.000000
25%,25008.750000,120.000000,1.000000
50%,50004.500000,1200.000000,1.000000
75%,75002.250000,3200.000000,1.000000
max,100000.000000,5500.000000,10.000000


In [4]:
##TRATAMENTO DE ERROS
#colunas
""" display(list(df_vendas.columns))
display(df_vendas["Loja"])  ##SÉRIE
display(df_vendas[["Loja", "Produto"]]) ##SOMENTE AS COLUNAS SELECIONADAS """

df_analise=df_vendas.drop(columns=["Data_Base"], errors='ignore')
display(df_analise.columns)

#nulos
#df_analise=df_vendas.dropna()  ##EXCLUI TODAS AS LINHAS COM VALORES NULOS (1 ou mais)
df_analise["Loja"]=df_analise["Loja"].fillna("Online")

#tipos de dados
df_analise["Data"]=pd.to_datetime(df_vendas["Data"], format=r'%Y-%m-%d')

#padronização
df_analise["Loja"]=df_analise["Loja"].str.strip()
df_analise["Loja"]=df_analise["Loja"].str.title()

#duplicatas
df_analise=df_analise.drop_duplicates(subset=["ID_Pedido", "Loja", "Produto"], keep='first') #subset->colunas avaliados / Keep->Manter

display(df_analise)
df_analise.info()

Index(['ID_Pedido', 'Data', 'Loja', 'Produto', 'Preco_Unitario', 'Qtd',
       'Cliente'],
      dtype='object')

,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente
0,1,2023-06-08,São Paulo,Mouse Gamer,120.0,1,Cliente_4095
1,2,2023-03-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_8750
2,3,2023-02-25,Online,"Monitor 27""",1200.0,1,Cliente_14859
3,4,2024-11-19,Rio De Janeiro,Mouse Gamer,120.0,2,Cliente_17343
4,5,2024-01-27,Rio De Janeiro,Smartphone Samsung,2200.0,1,Cliente_23377
...,...,...,...,...,...,...,...
99995,99996,2023-01-24,São Paulo,Mouse Gamer,120.0,2,Cliente_13732
99996,99997,2024-08-28,Rio De Janeiro,Cabo HDMI,40.0,1,Cliente_25058
99997,99998,2024-03-18,Curitiba,Smartphone Samsung,2200.0,2,Cliente_28864
99998,99999,2023-11-04,Porto Alegre,iPhone 14,5500.0,1,Cliente_4205


<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   ID_Pedido       100000 non-null  int64         
 1   Data            100000 non-null  datetime64[ns]
 2   Loja            100000 non-null  object        
 3   Produto         100000 non-null  object        
 4   Preco_Unitario  100000 non-null  float64       
 5   Qtd             100000 non-null  int64         
 6   Cliente         100000 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 6.1+ MB


In [5]:
##CRIANDO NOVAS COLUNAS:

#FATURAMENTO
df_analise["Faturamento"]=df_analise["Preco_Unitario"]*df_analise["Qtd"]

#FORMA DA VENDA
#np.where(condição, se for True, se for False)
df_analise["Forma_Venda"]=np.where(df_analise["Loja"]=="Online", "Online", "Presencial")

#REGIÃO
#display(df_analise["Loja"].unique()) ##MOSTRA OS VALORES únicos DENTRO DE UMA COLUNA
regioes={
    'São Paulo':'Sudeste',
    'Belo Horizonte': 'Sudeste', 
    'Online': 'Online', 
    'Rio De Janeiro': 'Sudeste',
    'Salvador': 'Nordeste', 
    'Recife': 'Nordeste', 
    'Curitiba': 'Sul', 
    'Porto Alegre': 'Sul'
}
df_analise["Região"]=df_analise["Loja"].map(regioes)  ##Para algo mais complexo, usar uma função no .map()

display(df_analise.head(10))

,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Faturamento,Forma_Venda,Região
0,1,2023-06-08,São Paulo,Mouse Gamer,120.0,1,Cliente_4095,120.0,Presencial,Sudeste
1,2,2023-03-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_8750,5500.0,Presencial,Sudeste
2,3,2023-02-25,Online,"Monitor 27""",1200.0,1,Cliente_14859,1200.0,Online,Online
3,4,2024-11-19,Rio De Janeiro,Mouse Gamer,120.0,2,Cliente_17343,240.0,Presencial,Sudeste
4,5,2024-01-27,Rio De Janeiro,Smartphone Samsung,2200.0,1,Cliente_23377,2200.0,Presencial,Sudeste
5,6,2023-03-11,São Paulo,Cabo HDMI,40.0,1,Cliente_24809,40.0,Presencial,Sudeste
6,7,2024-12-08,Salvador,"Monitor 27""",1200.0,1,Cliente_11076,1200.0,Presencial,Nordeste
7,8,2024-09-12,São Paulo,"Monitor 27""",1200.0,2,Cliente_14053,2400.0,Presencial,Sudeste
8,9,2023-05-20,Belo Horizonte,Mouse Gamer,120.0,1,Cliente_19175,120.0,Presencial,Sudeste
9,10,2024-10-16,Recife,Notebook HP,3200.0,1,Cliente_13508,3200.0,Presencial,Nordeste


In [6]:
display(df_analise.isna().sum())

ID_Pedido         0
Data              0
Loja              0
Produto           0
Preco_Unitario    0
Qtd               0
Cliente           0
Faturamento       0
Forma_Venda       0
Região            0
dtype: int64

In [7]:
##ANÁLISE->FILTRAR
##ORGANIZANDO:
df_analise=df_analise.sort_values(by=['Data', 'Loja'], ascending=True) ##Organiza by [ColunaX, ColunaY], ascending=True(ascendente True default)
#REINICIAR OS ÍNDICES (de 0 a ...)
df_analise=df_analise.reset_index(drop=True) ##Drop=True remove a antiga coluna de index


#.loc[ID, COLUNA_NAME] ->Filtra por meio do ID da linha, e Nome da coluna
cliente=df_analise.loc[4, "Cliente"]
produto=df_analise.loc[4, "Produto"]
loja=df_analise.loc[4, "Loja"]
print(cliente, produto, loja)

#.iloc[LINHA, COLUNA_NUM] ->Filtra por meio da posição da linha e coluna
cliente=df_analise.iloc[4, 6]
produto=df_analise.iloc[4, 3]
loja=df_analise.iloc[4, 2]
print(cliente, produto, loja) 

#Condicionais
id4=df_analise[df_analise['ID_Pedido']==4]
display(id4)
id4_cliente=df_analise.loc[df_analise['ID_Pedido']==4, "Cliente"].values[0] ##.values[0] ->Retorna um valor e não uma tabela
print(id4_cliente)

##EXPORTAR PEDAÇOS:
df_sp=df_analise[df_analise["Loja"]=="São Paulo"]
#df_sp.to_excel("Vendas_SP.xlsx", index=False)->EXPORTAR PARA .xlsx | index=False para não retornar os índices

#Vendas de 2024:
vendas_2024=df_analise[df_analise["Data"].dt.year==2024]

#Duplas condições | Vendas de cabos HDMI no Sul
x=df_analise["Produto"]=="Cabo HDMI"
y=df_analise["Região"]=="Sul"
vendas_hdmi_sul=df_analise[x & y]  ##&->and   |->or
#vendas_hdmi_sul=df_analise[(df_analise["Produto"]=="Cabo HDMI") & (df_analise["Região"]=="Sul" )]

display(vendas_hdmi_sul)
display(df_analise)

Cliente_29481 Mouse Gamer Belo Horizonte
Cliente_29481 Mouse Gamer Belo Horizonte


,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Faturamento,Forma_Venda,Região
94213,4,2024-11-19,Rio De Janeiro,Mouse Gamer,120.0,2,Cliente_17343,240.0,Presencial,Sudeste


Cliente_17343


,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Faturamento,Forma_Venda,Região
23,42051,2023-01-01,Curitiba,Cabo HDMI,40.0,10,Cliente_16622,400.0,Presencial,Sul
30,97659,2023-01-01,Curitiba,Cabo HDMI,40.0,1,Cliente_58,40.0,Presencial,Sul
35,1700,2023-01-01,Porto Alegre,Cabo HDMI,40.0,1,Cliente_14435,40.0,Presencial,Sul
43,35599,2023-01-01,Porto Alegre,Cabo HDMI,40.0,1,Cliente_18680,40.0,Presencial,Sul
44,37130,2023-01-01,Porto Alegre,Cabo HDMI,40.0,1,Cliente_22886,40.0,Presencial,Sul
...,...,...,...,...,...,...,...,...,...,...
99886,35127,2024-12-30,Curitiba,Cabo HDMI,40.0,2,Cliente_20687,80.0,Presencial,Sul
99899,78584,2024-12-30,Curitiba,Cabo HDMI,40.0,1,Cliente_21307,40.0,Presencial,Sul
99915,47774,2024-12-30,Porto Alegre,Cabo HDMI,40.0,2,Cliente_14363,80.0,Presencial,Sul
99919,64473,2024-12-30,Porto Alegre,Cabo HDMI,40.0,2,Cliente_11598,80.0,Presencial,Sul


,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Faturamento,Forma_Venda,Região
0,4222,2023-01-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_19759,5500.0,Presencial,Sudeste
1,7200,2023-01-01,Belo Horizonte,Smartphone Samsung,2200.0,2,Cliente_13597,4400.0,Presencial,Sudeste
2,12848,2023-01-01,Belo Horizonte,Cabo HDMI,40.0,1,Cliente_4327,40.0,Presencial,Sudeste
3,16936,2023-01-01,Belo Horizonte,Mouse Gamer,120.0,1,Cliente_10912,120.0,Presencial,Sudeste
4,20156,2023-01-01,Belo Horizonte,Mouse Gamer,120.0,1,Cliente_29481,120.0,Presencial,Sudeste
...,...,...,...,...,...,...,...,...,...,...
99995,80511,2024-12-30,São Paulo,Notebook Dell,3500.0,1,Cliente_487,3500.0,Presencial,Sudeste
99996,86855,2024-12-30,São Paulo,Teclado Mecânico,250.0,2,Cliente_5097,500.0,Presencial,Sudeste
99997,91693,2024-12-30,São Paulo,iPhone 14,5500.0,1,Cliente_17362,5500.0,Presencial,Sudeste
99998,93754,2024-12-30,São Paulo,Notebook Dell,3500.0,1,Cliente_9966,3500.0,Presencial,Sudeste


In [8]:
##ANÁLISE POR AGRUPAMENTO
faturamento=df_analise[["Loja", "Faturamento"]].groupby("Loja").sum() #as_index=False-> Faz que não seja o novo index e sim 1 a ...
faturamento=faturamento.sort_values(by=["Faturamento"], ascending=False)
faturamento=faturamento.reset_index()
#faturamento["Faturamento"]=faturamento["Faturamento"].map("R${:,.2f}".format)->FORMATAR
display(faturamento)

,Loja,Faturamento
0,Salvador,42300610.0
1,Rio De Janeiro,42294720.0
2,Recife,42190390.0
3,São Paulo,42090690.0
4,Belo Horizonte,41714890.0
5,Porto Alegre,41678460.0
6,Curitiba,41121720.0
7,Online,6080850.0


In [9]:
##RANKING DE PRODUTOS QUE MAIS VENDERAM NO ONLINE
df_vendas_online=df_analise[df_analise["Loja"]=="Online"]
analise_vendas_online=df_vendas_online[["Produto", "Faturamento"]].groupby("Produto").sum()
analise_vendas_online=analise_vendas_online.sort_values("Faturamento", ascending=False)
analise_vendas_online["Faturamento"]=analise_vendas_online["Faturamento"].map("R${:,.2f}".format)
analise_vendas_online=analise_vendas_online.reset_index()

analise_vendas_online=analise_vendas_online.rename(columns={"Faturamento":"Dinheiro"}) #.rename(columns={"nome_antigo":"nome_novo"})

display(analise_vendas_online)

,Produto,Dinheiro
0,iPhone 14,"R$2,145,000.00"
1,Notebook HP,"R$1,414,400.00"
2,Notebook Dell,"R$1,291,500.00"
3,Smartphone Samsung,"R$684,200.00"
4,"Monitor 27""","R$398,400.00"
5,Teclado Mecânico,"R$85,750.00"
6,Mouse Gamer,"R$45,480.00"
7,Cabo HDMI,"R$16,120.00"


In [10]:
##RANKING POR LOJA E PRODUTO

produtos_per_loja=df_analise[["Loja", "Produto", "Qtd"]].groupby(["Loja", "Produto"], as_index=False).sum()
#produtos_per_loja.to_excel("Loja_Produto.xlsx")
display(produtos_per_loja)

##RANKIN DE PRODUTO LOJA
loja_per_produto=produtos_per_loja=df_analise[['Loja', 'Produto', 'Qtd']].groupby(['Produto', 'Loja'], as_index=False).sum()
display(loja_per_produto)

,Loja,Produto,Qtd
0,Belo Horizonte,Cabo HDMI,2636
1,Belo Horizonte,"Monitor 27""",2625
2,Belo Horizonte,Mouse Gamer,2465
3,Belo Horizonte,Notebook Dell,2654
4,Belo Horizonte,Notebook HP,2775
...,...,...,...
59,São Paulo,Notebook Dell,2535
60,São Paulo,Notebook HP,2592
61,São Paulo,Smartphone Samsung,2476
62,São Paulo,Teclado Mecânico,2589


,Produto,Loja,Qtd
0,Cabo HDMI,Belo Horizonte,2636
1,Cabo HDMI,Curitiba,2698
2,Cabo HDMI,Online,403
3,Cabo HDMI,Porto Alegre,2571
4,Cabo HDMI,Recife,2534
...,...,...,...
59,iPhone 14,Porto Alegre,2540
60,iPhone 14,Recife,2652
61,iPhone 14,Rio De Janeiro,2702
62,iPhone 14,Salvador,2671


In [11]:
# Transforma os produtos em colunas para comparar as lojas lado a lado
pivot_lojas = produtos_per_loja.pivot(index="Loja", columns="Produto", values="Qtd").fillna(0)
display(pivot_lojas)

# Transforma as lojas em colunas para comparar os produtos lado a lado
pivot_produtos = produtos_per_loja.pivot(index="Produto", columns="Loja", values="Qtd").fillna(0)
display(pivot_produtos)

Produto,Cabo HDMI,"Monitor 27""",Mouse Gamer,Notebook Dell,Notebook HP,Smartphone Samsung,Teclado Mecânico,iPhone 14
Loja,,,,,,,,
Belo Horizonte,2636,2625,2465,2654,2775,2597,2609,2478
Curitiba,2698,2626,2600,2517,2529,2444,2742,2652
Online,403,332,379,369,442,311,343,390
Porto Alegre,2571,2655,2611,2770,2511,2603,2598,2540
Recife,2534,2647,2639,2566,2651,2660,2775,2652
Rio De Janeiro,2747,2798,2652,2534,2614,2626,2548,2702
Salvador,2566,2545,2711,2512,2785,2627,2579,2671
São Paulo,2649,2796,2714,2535,2592,2476,2589,2735


Loja,Belo Horizonte,Curitiba,Online,Porto Alegre,Recife,Rio De Janeiro,Salvador,São Paulo
Produto,,,,,,,,
Cabo HDMI,2636,2698,403,2571,2534,2747,2566,2649
"Monitor 27""",2625,2626,332,2655,2647,2798,2545,2796
Mouse Gamer,2465,2600,379,2611,2639,2652,2711,2714
Notebook Dell,2654,2517,369,2770,2566,2534,2512,2535
Notebook HP,2775,2529,442,2511,2651,2614,2785,2592
Smartphone Samsung,2597,2444,311,2603,2660,2626,2627,2476
Teclado Mecânico,2609,2742,343,2598,2775,2548,2579,2589
iPhone 14,2478,2652,390,2540,2652,2702,2671,2735


In [41]:
##GERENTES QUE BATERAM A META EM JANEIRO/2023
#display(df_analise.head())


jan_2023=df_analise[(df_analise["Data"].dt.year==2023) & (df_analise["Data"].dt.month==1)]
jan_2023=jan_2023[["Loja", "Faturamento"]].groupby("Loja", as_index=False).sum()
jan_2023=jan_2023.merge(df_gerentes, on="Loja", how="left")
jan_2023["Bateu_Meta"]=np.where(jan_2023["Faturamento"]>=jan_2023["Meta_Mensal"], "Sim", "Não")
jan_2023=jan_2023.dropna()

##JUNTAR BASES DE DADOS
#merge()->informações diferentes | pega uma tabela e coloca do lado da outra  ##INNER JOIN
#concat()->Pega uma tabela e coloca em baixo da outra



display(jan_2023)

,Loja,Faturamento,Gerente,Meta_Mensal,Bateu_Meta
0,Belo Horizonte,1779100.0,Juliana,55000.0,Sim
1,Curitiba,1986920.0,Roberto,45000.0,Sim
3,Porto Alegre,1726640.0,Pedro,42000.0,Sim
4,Recife,1779020.0,Marcos,48000.0,Sim
5,Rio De Janeiro,1736830.0,Fernanda,60000.0,Sim
6,Salvador,1686070.0,Ana,52000.0,Sim
7,São Paulo,1831140.0,Carlos,50000.0,Sim


In [48]:
##ANÁLISE TEMPORAL⏱️
df_analise["Mes-Ano"]=df_analise["Data"].dt.to_period("M")
vendas_mes=df_analise[["Mes-Ano", "Faturamento"]].groupby("Mes-Ano", as_index=False).sum()
vendas_mes["Faturamento"]=vendas_mes["Faturamento"].map("R${:,.2f}".format)
display(vendas_mes)

,Mes-Ano,Faturamento
0,2023-01,"R$12,930,290.00"
1,2023-02,"R$11,515,150.00"
2,2023-03,"R$12,516,080.00"
3,2023-04,"R$12,528,900.00"
4,2023-05,"R$12,940,470.00"
5,2023-06,"R$12,455,820.00"
6,2023-07,"R$12,550,990.00"
7,2023-08,"R$12,989,130.00"
8,2023-09,"R$12,118,180.00"
9,2023-10,"R$12,918,990.00"
